In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# =============================================================================
# INPUT FILES
# =============================================================================

HAZARD_CSV = Path("data") / "hazard.csv"
EXPOSURE_CSV = Path("data") / "exposure_district.csv"
VULNERABILITY_CSV = Path("data") / "vulnerability.csv"
GOVT_CSV = Path("data") / "government_response_district.csv"
MASTER_CSV = Path("data") / "MASTER_VARIABLES.csv"

hazard = pd.read_csv(HAZARD_CSV)
exposure = pd.read_csv(EXPOSURE_CSV)
vulnerability = pd.read_csv(VULNERABILITY_CSV)
gov = pd.read_csv(GOVT_CSV)

# =============================================================================
# STANDARDIZE COLUMN NAMES
# =============================================================================

hazard.columns = hazard.columns.str.replace("_", "-", regex=False)
exposure.columns = exposure.columns.str.replace("_", "-", regex=False)
vulnerability.columns = vulnerability.columns.str.replace("_", "-", regex=False)
gov.columns = gov.columns.str.replace("_", "-", regex=False)

# =============================================================================
# KEEP REQUIRED COLUMNS
# =============================================================================

hazard = hazard[
    ["dtname", "timeperiod", "heat-hazard", "heat-days-score"]
]

exposure = exposure[
    ["dtname", "timeperiod", "exposure"]
]

vulnerability = vulnerability[
    ["dtname", "timeperiod", "vulnerability"]
]

gov = gov[
    ["dtname", "timeperiod", "government-response"]
]

# =============================================================================
# MERGE COMPONENTS
# =============================================================================

df = hazard.merge(
    exposure,
    on=["dtname", "timeperiod"],
    how="inner"
)

df = df.merge(
    vulnerability,
    on=["dtname", "timeperiod"],
    how="inner"
)

df = df.merge(gov, on=["dtname", "timeperiod"], how="inner")

# =============================================================================
# CLEAN KEYS
# =============================================================================

df["dtname"] = (
    df["dtname"]
    .astype(str)
    .str.strip()
)

df["timeperiod"] = (
    df["timeperiod"]
    .astype(str)
    .str.strip()
)

# =============================================================================
# TOPSIS WEIGHTS
# =============================================================================

weights = {
    "heat-hazard": 4,
    "exposure": 1,
    "vulnerability": 2,
    "government-response": 2,
}

total_weight = sum(weights.values())

# =============================================================================
# TOPSIS BY MONTH
# =============================================================================

results = []

for tp, g in df.groupby("timeperiod"):

    g = g.copy()

    norm = pd.DataFrame(index=g.index)

    # -------------------------------------------------------------------------
    # MIN-MAX NORMALIZATION
    # -------------------------------------------------------------------------

    for col in weights:

        min_v = g[col].min()
        max_v = g[col].max()

        if max_v == min_v:
            norm[col] = 0
        else:
            norm[col] = (
                (g[col] - min_v)
                /
                (max_v - min_v)
            )

    # -------------------------------------------------------------------------
    # APPLY WEIGHTS
    # -------------------------------------------------------------------------

    for col in weights:
        norm[col] = (
            norm[col]
            *
            (weights[col] / total_weight)
        )

    # -------------------------------------------------------------------------
    # IDEAL BEST / WORST
    # -------------------------------------------------------------------------

    ideal_best = norm.max()
    ideal_worst = norm.min()

    # -------------------------------------------------------------------------
    # DISTANCES
    # -------------------------------------------------------------------------

    dist_best = np.sqrt(
        ((norm - ideal_best) ** 2).sum(axis=1)
    )

    dist_worst = np.sqrt(
        ((norm - ideal_worst) ** 2).sum(axis=1)
    )

    # -------------------------------------------------------------------------
    # TOPSIS SCORE
    # -------------------------------------------------------------------------

    g["topsis-score"] = (
        dist_worst
        /
        (dist_best + dist_worst)
    )

    results.append(g)

# =============================================================================
# COMBINE RESULTS
# =============================================================================

df = pd.concat(
    results,
    ignore_index=True
)

# =============================================================================
# RISK CLASSIFICATION
# =============================================================================

def classify(score):

    if score <= 0.20:
        return 1
    elif score <= 0.40:
        return 2
    elif score <= 0.60:
        return 3
    elif score <= 0.80:
        return 4
    else:
        return 5


df["heat-risk-score"] = (
    df["topsis-score"]
    .apply(classify)
)

# =============================================================================
# SUMMARY
# =============================================================================

print("\nTOPSIS Summary")
print(df["topsis-score"].describe())

print("\nRisk Class Distribution")
print(
    df["heat-risk-score"]
    .value_counts()
    .sort_index()
)

print("\nPreview")
print(
    df[
        [
            "dtname",
            "timeperiod",
            "topsis-score",
            "heat-risk-score"
        ]
    ].head()
)


# =============================================================================
# APPEND RESULTS TO MASTER_VARIABLES
# =============================================================================

# convert district key name
df = df.rename(
    columns={
        "dtname": "district"
    }
)

master = pd.read_csv(MASTER_CSV)
fy_cumsum = pd.read_csv(GOVT_CSV)

master.columns = (
    master.columns
    .str.replace("_", "-", regex=False)
)

fy_cumsum.columns = (
    fy_cumsum.columns
    .str.replace("_", "-", regex=False)
)

# -----------------------------------------------------------------------------
# CLEAN KEYS
# -----------------------------------------------------------------------------

master["district"] = (
    master["district"]
    .astype(str)
    .str.strip()
)

master["timeperiod"] = (
    master["timeperiod"]
    .astype(str)
    .str.strip()
)

fy_cumsum["dtname"] = (
    fy_cumsum["dtname"]
    .astype(str)
    .str.strip()
)

fy_cumsum["timeperiod"] = (
    fy_cumsum["timeperiod"]
    .astype(str)
    .str.strip()
)

fy_cumsum = fy_cumsum.rename(
    columns={
        "dtname": "district"
    }
)

# -----------------------------------------------------------------------------
# REMOVE OLD COLUMNS
# -----------------------------------------------------------------------------

for col in [
    "heat-hazard",
    "exposure",
    "vulnerability",
    "government-response",
    "topsis-score",
    "heat-risk-score",
]:
    if col in master.columns:
        master = master.drop(columns=col)

# -----------------------------------------------------------------------------
# MERGE TOPSIS RESULTS
# -----------------------------------------------------------------------------

final_df = master.merge(
    df[
        [
            "district",
            "timeperiod",
            "heat-hazard",
            "exposure",
            "vulnerability",
            "government-response",
            "topsis-score",
            "heat-risk-score",
        ]
    ],
    on=["district", "timeperiod"],
    how="left",
)

# -----------------------------------------------------------------------------
# MERGE FY CUMULATIVE TENDER VALUES
# -----------------------------------------------------------------------------

if "cum-tender-value" in fy_cumsum.columns:

    final_df = final_df.merge(
        fy_cumsum[
            [
                "district",
                "timeperiod",
                "cum-tender-value",
            ]
        ],
        on=["district", "timeperiod"],
        how="left",
    )

# =============================================================================
# SAVE RC LEVEL OUTPUT
# =============================================================================

OUTPUT = Path("data") / "archive" / "final_risk_score.csv"

final_df.to_csv(
    OUTPUT,
    index=False
)

print(f"\nSaved final file: {OUTPUT}")
print(f"Rows: {len(final_df)}")
print(f"Columns: {len(final_df.columns)}")

# =============================================================================
# DISTRICT LEVEL AGGREGATION
# =============================================================================

district_df = (
    final_df.groupby(
        ["district", "timeperiod"],
        as_index=False
    )
    .agg(
        {
            "dtname": "first",

            "health-centres-count": "sum",

            "cum-tender-value": "first",
            "total-tender-awarded-value": "sum",

            "sum-aged-population": "sum",
            "sum-young-population": "sum",
            "sum-population": "sum",

            "avg-electricity": "mean",

            "rc-piped-hhds-pct": "mean",
            "rc-nosanitation-hhds-pct": "mean",

            "workers-affected-pct": "mean",
            "pct-ncd": "mean",

            "land-surface-temperature": "mean",
            "land-surface-temperature-raster": "first",

            "heat-days-score": "mean",

            "heat-hazard": "first",
            "exposure": "first",
            "vulnerability": "first",
            "government-response": "first",

            "topsis-score": "first",
            "heat-risk-score": "first",
        }
    )
)

# =============================================================================
# OBJECT ID LOOKUP
# =============================================================================

object_lookup = (
    master[
        [
            "district",
            "object-id"
        ]
    ]
    .drop_duplicates(
        subset="district"
    )
)

district_df = district_df.merge(
    object_lookup,
    on="district",
    how="left"
)


# =============================================================================
# RENAME OUTPUT COLUMNS
# =============================================================================

district_df = district_df.rename(
    columns={
        "object_id":
            "object-id",

        "rc-piped-hhds-pct":
            "piped-hhds-pct",

        "rc-nosanitation-hhds-pct":
            "nosanitation-hhds-pct",

        "cum-tender-value":
            "total-tender-awarded-value-fy-cumsum",
    }
)

# Keep only the first two parts of the object_id to represent only the district
district_df["object-id"] = (
    district_df["object-id"]
    .astype(str)
    .str.rsplit("-", n=1)
    .str[0]
)

# =============================================================================
# ROUNDING
# =============================================================================

numeric_cols = district_df.select_dtypes(
    include=np.number
).columns

district_df[numeric_cols] = (
    district_df[numeric_cols]
    .round(3)
)

# rounding off
# Population columns as integers
int_cols = [
    "sum-aged-population",
    "sum-young-population",
    "sum-population",
]

district_df[int_cols] = district_df[int_cols].round().astype("Int64")

# All other numeric columns to 2 decimal places
numeric_cols = district_df.select_dtypes(include="number").columns
decimal_cols = numeric_cols.difference(int_cols)

# =============================================================================
# SAVE DISTRICT OUTPUT
# =============================================================================

DISTRICT_OUTPUT = (
    Path("data")
    / "district_final_risk_score.csv"
)

district_df.to_csv(
    DISTRICT_OUTPUT,
    index=False
)

print(f"\nSaved district file: {DISTRICT_OUTPUT}")
print(f"Rows: {len(district_df)}")
print(f"Columns: {len(district_df.columns)}")

print("\nDistrict Preview")

print(
    district_df[
        [
            "district",
            "timeperiod",
            "government-response",
            "topsis-score",
            "heat-risk-score",
        ]
    ].head()
)


TOPSIS Summary
count    2275.000000
mean        0.509394
std         0.140179
min         0.048021
25%         0.394879
50%         0.500000
75%         0.612034
max         0.895222
Name: topsis-score, dtype: float64

Risk Class Distribution
heat-risk-score
1      18
2     565
3    1024
4     616
5      52
Name: count, dtype: int64

Preview
       dtname timeperiod  topsis-score  heat-risk-score
0      BAJALI    2021_01      0.414286                3
1       BAKSA    2021_01      0.414286                3
2     BARPETA    2021_01      0.429641                3
3   BISWANATH    2021_01      0.307713                2
4  BONGAIGAON    2021_01      0.421429                3

Saved final file: data/archive/final_risk_score.csv
Rows: 11700
Columns: 37

Saved district file: data/district_final_risk_score.csv
Rows: 2275
Columns: 24

District Preview
  district timeperiod  government-response  topsis-score  heat-risk-score
0   BAJALI    2021_01                    3         0.414              